# Section 2 · Notebook 1 — Building Tools

1. From function to tool — turning a Python function into a tool with `@tool`
2. Your first tool-enabled agent — a tip calculator
3. When one tool isn't enough — a multi-tool sales assistant
4. Building custom tools — an inventory checker

In [1]:
%pip install -q -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from strands.models.bedrock import BedrockModel

MODEL_ID = os.getenv("BEDROCK_MODEL_ID", "global.anthropic.claude-sonnet-4-6")
REGION = os.getenv("AWS_REGION", os.getenv("AWS_DEFAULT_REGION", "us-east-1"))
os.environ.setdefault("AWS_REGION", REGION)

model = BedrockModel(model_id=MODEL_ID, region_name=REGION)
print(f"Using {MODEL_ID} in {REGION}")

Using global.anthropic.claude-sonnet-4-6 in us-east-1


---
## 1. From Function to Tool

Here is an ordinary Python function. It works when *you* call it, but an agent has no way to know it exists:

In [3]:
import requests


def check_server_status_plain(server_url):
    """Check if a server is responding."""
    try:
        response = requests.get(server_url, timeout=5)
        return f"Server is up. Status code: {response.status_code}"
    except requests.exceptions.RequestException:
        return "Server is down or unreachable"


print(check_server_status_plain("https://httpbin.org/get"))

Server is up. Status code: 200


Add three things and it becomes a tool the agent can discover and call:
- the `@tool` decorator
- type hints
- a clear docstring (with an `Args:` section)

In [4]:
from strands import Agent, tool


@tool  # Decorator
def check_server_status(server_url: str) -> str:  # Type hints
    """Check if a server is responding by making an HTTP request.

    Args:
        server_url: The URL of the server to check

    Returns:
        A message indicating whether the server is up or down
    """
    try:
        response = requests.get(server_url, timeout=5)
        return f"Server is up. Status code: {response.status_code}"
    except requests.exceptions.RequestException:
        return "Server is down or unreachable"

In [5]:
agent = Agent(model=model, tools=[check_server_status])
response = agent("Is the staging server running? Check https://httpbin.org/get")

Sure! Let me check the staging server for you right now.
Tool #1: check_server_status
The staging server at **https://httpbin.org/get** is **up and running**! ✅ It responded with a **200 OK** status code, meaning it's healthy and accessible. No issues to report!

---
## 2. Your First Tool-Enabled Agent — Tip Calculator

In [6]:
@tool
def calculate_tip(bill_amount: float, tip_percentage: float, num_people: int = 1) -> dict:
    """Calculate tip and split the bill among people.

    Args:
        bill_amount: Total bill amount in dollars
        tip_percentage: Tip percentage (e.g., 15, 18, 20)
        num_people: Number of people splitting the bill (default: 1)
    """
    tip = bill_amount * (tip_percentage / 100)
    total = bill_amount + tip
    per_person = total / num_people

    return {
        "bill": bill_amount,
        "tip": round(tip, 2),
        "total": round(total, 2),
        "per_person": round(per_person, 2),
    }


agent = Agent(model=model, tools=[calculate_tip])

response = agent(
    "The bill is $85. What's a 20% tip, and how much does each person pay if we're splitting it 4 ways?"
)
print("\n\nFinal answer:\n", response.message["content"][0]["text"])

Sure! Let me calculate that for you right away!
Tool #1: calculate_tip
Here's the breakdown for your $85 bill:

| | Amount |
|---|---|
| 💵 Original Bill | $85.00 |
| 🙏 Tip (20%) | $17.00 |
| 🧾 Total | $102.00 |
| 👤 Per Person (÷4) | **$25.50** |

Each person owes **$25.50**. Nice and even! 🎉

Final answer:
 Here's the breakdown for your $85 bill:

| | Amount |
|---|---|
| 💵 Original Bill | $85.00 |
| 🙏 Tip (20%) | $17.00 |
| 🧾 Total | $102.00 |
| 👤 Per Person (÷4) | **$25.50** |

Each person owes **$25.50**. Nice and even! 🎉


Try different phrasings — the agent understands intent, not just keywords:

In [7]:
agent("What's a 15% tip on $42?")
agent("Bill is $120, we want to tip 18%, split between 3 people")
agent("Calculate tip for $67.50 at 20%")

Sure! Let me calculate that for you!
Tool #2: calculate_tip
Here's the breakdown for your $42 bill:

| | Amount |
|---|---|
| 💵 Original Bill | $42.00 |
| 🙏 Tip (15%) | $6.30 |
| 🧾 Total | $48.30 |

A 15% tip on $42 comes out to **$6.30**, bringing your total to **$48.30**. Enjoy your meal! 🍽️On it!
Tool #3: calculate_tip
Here's the breakdown for your $120 bill:

| | Amount |
|---|---|
| 💵 Original Bill | $120.00 |
| 🙏 Tip (18%) | $21.60 |
| 🧾 Total | $141.60 |
| 👤 Per Person (÷3) | **$47.20** |

Each person owes **$47.20**. Enjoy your meal! 🍽️Sure!
Tool #4: calculate_tip
Here's the breakdown for your $67.50 bill:

| | Amount |
|---|---|
| 💵 Original Bill | $67.50 |
| 🙏 Tip (20%) | $13.50 |
| 🧾 Total | **$81.00** |

A 20% tip on $67.50 comes out to **$13.50**, bringing your total to **$81.00**. Enjoy! 🎉

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': "Here's the breakdown for your $67.50 bill:\n\n| | Amount |\n|---|---|\n| 💵 Original Bill | $67.50 |\n| 🙏 Tip (20%) | $13.50 |\n| 🧾 Total | **$81.00** |\n\nA 20% tip on $67.50 comes out to **$13.50**, bringing your total to **$81.00**. Enjoy! 🎉"}], 'metadata': {'usage': {'inputTokens': 1612, 'outputTokens': 114, 'totalTokens': 1726}, 'metrics': {'latencyMs': 1340, 'timeToFirstByteMs': 1081}}, 'tracking_id': '5b7175d4-1a35-47c8-a791-4ec3c3e2639e'}, metrics=EventLoopMetrics(cycle_count=8, tool_metrics={'calculate_tip': ToolMetrics(tool={'toolUseId': 'tooluse_eIThVGBbJW08aTrFAXt1nI', 'name': 'calculate_tip', 'input': {'bill_amount': 67.5, 'tip_percentage': 20}}, call_count=4, success_count=4, error_count=0, total_time=0.003780841827392578)}, cycle_durations=[1.665578842163086, 3.4040772914886475, 1.427603006362915, 1.6299750804901123, 1.542405128479004, 1.496337890625, 1.417268991470337, 1.5536799430847

---
## 3. When One Tool Isn't Enough — Multi-Tool Sales Assistant

Give the agent several small tools and it works out the order to call them in. One request — "pull sales data and email a summary" — becomes three tool calls.

In [8]:
@tool
def get_sales_data(quarter: str) -> dict:
    """Retrieve sales data for a specific quarter."""
    return {"revenue": 1250000, "deals": 47, "quarter": quarter}


@tool
def analyze_sales(revenue: int, deals: int, quarter: str) -> str:
    """Calculate key metrics from sales data."""
    avg_deal = revenue / deals
    return f"{quarter}: ${revenue:,} revenue, {deals} deals, ${avg_deal:,.0f} avg deal size"


@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email message."""
    print(f"\n[EMAIL] to={to} | subject={subject}\n{body}\n")
    return f"Email sent to {to}"


agent = Agent(model=model, tools=[get_sales_data, analyze_sales, send_email])
response = agent("Pull last quarter's sales data and email a summary to team@company.com")

I'll start by pulling last quarter's sales data right away!
Tool #1: get_sales_data
Got the data! Now I'll simultaneously analyze the sales metrics and — since both can be done in parallel — I'll wait on the analysis before sending the email to ensure the summary is accurate. Let me run the analysis now!
Tool #2: analyze_sales
Great, analysis complete! Now I'll send the summary email to the team.
Tool #3: send_email

[EMAIL] to=team@company.com | subject=Q1 2025 Sales Summary
Hi Team,

Here's a summary of our Q1 2025 sales performance:

- 💰 Total Revenue: $1,250,000
- 🤝 Total Deals Closed: 47
- 📊 Average Deal Size: $26,596

Great work this quarter! Let's keep up the momentum heading into Q2.

Best regards

Everything is done! Here's a recap of what was completed:

1. ✅ **Data Retrieved** – Pulled Q1 2025 sales data successfully.
2. ✅ **Analysis Completed** – Key metrics were calculated from the raw data.
3. ✅ **Email Sent** – A summary was delivered to team@company.com.

**Q1 2025 High

---
## 4. Building Custom Tools — Inventory Checker

When no prebuilt tool fits, build your own. This one simulates querying a product database.

In [9]:
@tool
def check_inventory(product_id: str) -> str:
    """Check if a product is in stock.

    Args:
        product_id: The product ID to check (e.g., "PROD-123")
    """
    inventory = {"PROD-123": 15, "PROD-456": 0, "PROD-789": 8}
    quantity = inventory.get(product_id, 0)

    if quantity > 0:
        return f"Product {product_id} is in stock. We have {quantity} units available."
    return f"Product {product_id} is currently out of stock."


agent = Agent(model=model, tools=[check_inventory])

# All of these work — the agent understands intent, not just keywords
agent("Is PROD-123 in stock?")
agent("Do we have PROD-456 available?")
agent("Check inventory for PROD-789")
agent("Can I order PROD-123 right now?")


Tool #1: check_inventory
Yes, **PROD-123** is currently in stock! There are **15 units** available. Let me know if you need any further assistance!
Tool #2: check_inventory
Unfortunately, **PROD-456** is currently **out of stock**. You may want to check back later or consider an alternative product. Let me know if there's anything else I can help with!
Tool #3: check_inventory
Great news! **PROD-789** is currently **in stock** with **8 units** available. Let me know if you have any other questions!Based on the inventory check we already performed, **PROD-123** is **in stock** with **15 units** available, so it should be orderable right now!

However, I don't have the ability to place orders. You'll need to use your ordering system or contact the appropriate team to proceed. Let me know if there's anything else I can help with!

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': "Based on the inventory check we already performed, **PROD-123** is **in stock** with **15 units** available, so it should be orderable right now!\n\nHowever, I don't have the ability to place orders. You'll need to use your ordering system or contact the appropriate team to proceed. Let me know if there's anything else I can help with!"}], 'metadata': {'usage': {'inputTokens': 1004, 'outputTokens': 82, 'totalTokens': 1086}, 'metrics': {'latencyMs': 1553, 'timeToFirstByteMs': 1036}}, 'tracking_id': 'cd918ef7-8740-4ac1-a526-72a64a8f46b9'}, metrics=EventLoopMetrics(cycle_count=7, tool_metrics={'check_inventory': ToolMetrics(tool={'toolUseId': 'tooluse_liojMgtyYhMBz5VYQOBBvp', 'name': 'check_inventory', 'input': {'product_id': 'PROD-789'}}, call_count=3, success_count=3, error_count=0, total_time=0.0015332698822021484)}, cycle_durations=[1.3805341720581055, 1.4334440231323242, 3.7874879837036133, 1.4730